# Arousal Prediction v7

**Key fixes from v6:**
- v6 removed calibration → score dropped 0.23→0.20. Calibration is ESSENTIAL
- Test PIDs are completely different from train PIDs → need cross-person features

**v7 strategy:**
- Keep v4 calibration unchanged
- Add interaction features (EDA×HR, brain ratios) — person-independent
- Add XGBoost to ensemble (RF + LGB + XGB)
- Keep everything else from v4

In [4]:
!pip install -q pandas numpy scipy scikit-learn lightgbm xgboost imbalanced-learn matplotlib


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.signal import welch
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
import xgboost as xgb
from imblearn.over_sampling import ADASYN, SMOTE
import matplotlib.pyplot as plt
np.random.seed(42)
print('Libraries loaded.')

Libraries loaded.


## 1. Load Data

In [6]:
DATA = '.'
train_labels = pd.read_csv(f'{DATA}/train-label.csv')
test_labels  = pd.read_csv(f'{DATA}/test-label.csv')
trainbvp   = pd.read_csv(f'{DATA}/train-bvp.csv')
traineda   = pd.read_csv(f'{DATA}/train-eda.csv')
traintemp  = pd.read_csv(f'{DATA}/train-temp.csv')
trainhr    = pd.read_csv(f'{DATA}/train-hr.csv')
trainibi   = pd.read_csv(f'{DATA}/train-ibi.csv')
trainbrain = pd.read_csv(f'{DATA}/train-brain.csv')
trainacc   = pd.read_csv(f'{DATA}/train-acc.csv')
testbvp    = pd.read_csv(f'{DATA}/test-bvp.csv')
testeda    = pd.read_csv(f'{DATA}/test-eda.csv')
testtemp   = pd.read_csv(f'{DATA}/test-temp.csv')
testhr     = pd.read_csv(f'{DATA}/test-hr.csv')
testibi    = pd.read_csv(f'{DATA}/test-ibi.csv')
testbrain  = pd.read_csv(f'{DATA}/test-brain.csv')
testacc    = pd.read_csv(f'{DATA}/test-acc.csv')
print(f'Train: {len(train_labels)} | Test: {len(test_labels)}')
print(train_labels['arousal'].value_counts().sort_index())

Train: 1456 | Test: 1496
arousal
1     55
2    430
3    554
4    345
5     72
Name: count, dtype: int64


## 2. Feature Helpers (same as v4)

In [7]:
WINDOW       = 5000
BVP_WINDOW   = 10000
BRAIN_WINDOW = 10000
IBI_WINDOW   = 20000
BASELINE_W   = 30000

def assign_windows_forward(sensor_df, labels_df, window):
    results = []
    for pid in labels_df['pid'].unique():
        lbl = labels_df[labels_df['pid']==pid].sort_values('timestamp')
        sen = sensor_df[sensor_df['pid']==pid].copy()
        if len(sen) == 0: continue
        lbl_ts = lbl['timestamp'].values
        bins = np.append(lbl_ts, lbl_ts[-1] + window)
        sen['label_ts'] = pd.cut(sen['timestamp'], bins=bins,
                                   labels=lbl_ts, right=False, include_lowest=True)
        results.append(sen.dropna(subset=['label_ts']))
    if not results: return pd.DataFrame()
    out = pd.concat(results, ignore_index=True)
    out['label_ts'] = out['label_ts'].astype(np.int64)
    return out

def assign_windows_lookback(sensor_df, labels_df, window):
    results = []
    for pid in labels_df['pid'].unique():
        lbl = labels_df[labels_df['pid']==pid].sort_values('timestamp')
        sen = sensor_df[sensor_df['pid']==pid].copy()
        if len(sen) == 0: continue
        for ts in lbl['timestamp'].values:
            chunk = sen[(sen['timestamp'] >= ts-window) & (sen['timestamp'] < ts)]
            if len(chunk) > 0:
                chunk = chunk.copy()
                chunk['label_ts'] = ts
                chunk['pid'] = pid
                results.append(chunk)
    if not results: return pd.DataFrame()
    out = pd.concat(results, ignore_index=True)
    out['label_ts'] = out['label_ts'].astype(np.int64)
    return out

def safe_stats(vals, prefix):
    a = np.array(vals, dtype=float); a = a[~np.isnan(a)]
    if len(a) == 0:
        return {f'{prefix}_{k}': np.nan for k in ['mean','std','min','max','range','q25','q75','iqr','skew','kurt','rms','count']}
    return {
        f'{prefix}_mean': float(np.mean(a)),
        f'{prefix}_std':  float(np.std(a)) if len(a)>1 else 0.0,
        f'{prefix}_min':  float(np.min(a)),
        f'{prefix}_max':  float(np.max(a)),
        f'{prefix}_range':float(np.ptp(a)),
        f'{prefix}_q25':  float(np.percentile(a,25)),
        f'{prefix}_q75':  float(np.percentile(a,75)),
        f'{prefix}_iqr':  float(np.percentile(a,75)-np.percentile(a,25)),
        f'{prefix}_skew': float(stats.skew(a)) if len(a)>2 else 0.0,
        f'{prefix}_kurt': float(stats.kurtosis(a)) if len(a)>2 else 0.0,
        f'{prefix}_rms':  float(np.sqrt(np.mean(a**2))),
        f'{prefix}_count':float(len(a)),
    }

def spectral_features(vals, fs, prefix):
    a = np.array(vals, dtype=float); a = a[~np.isnan(a)]
    if len(a) < 8:
        return {f'{prefix}_lf': np.nan, f'{prefix}_hf': np.nan, f'{prefix}_lf_hf': np.nan}
    try:
        f, psd = welch(a, fs=fs, nperseg=min(len(a), 64))
        lf = np.trapz(psd[(f>=0.04)&(f<=0.15)], f[(f>=0.04)&(f<=0.15)])
        hf = np.trapz(psd[(f>=0.15)&(f<=0.40)], f[(f>=0.15)&(f<=0.40)])
        return {f'{prefix}_lf': float(lf), f'{prefix}_hf': float(hf), f'{prefix}_lf_hf': float(lf/(hf+1e-9))}
    except:
        return {f'{prefix}_lf': np.nan, f'{prefix}_hf': np.nan, f'{prefix}_lf_hf': np.nan}

print('Helpers defined.')

Helpers defined.


## 3. Feature Extraction (v4 base + v7 interaction features)

In [8]:
def build_features(labels_df, bvp_df, eda_df, temp_df, hr_df, ibi_df, brain_df, acc_df):
    print(' Forward windows...')
    bvp_w   = assign_windows_forward(bvp_df,   labels_df, BVP_WINDOW)
    eda_w   = assign_windows_forward(eda_df,   labels_df, WINDOW)
    temp_w  = assign_windows_forward(temp_df,  labels_df, WINDOW)
    hr_w    = assign_windows_forward(hr_df,    labels_df, WINDOW)
    ibi_w   = assign_windows_forward(ibi_df,   labels_df, IBI_WINDOW)
    brain_w = assign_windows_forward(brain_df, labels_df, BRAIN_WINDOW)
    acc_w   = assign_windows_forward(acc_df,   labels_df, WINDOW)

    print(' Lookback baseline...')
    eda_b  = assign_windows_lookback(eda_df,  labels_df, BASELINE_W)
    temp_b = assign_windows_lookback(temp_df, labels_df, BASELINE_W)
    hr_b   = assign_windows_lookback(hr_df,   labels_df, BASELINE_W)
    acc_b  = assign_windows_lookback(acc_df,  labels_df, BASELINE_W)

    # BVP
    print(' BVP...')
    bvp_agg = bvp_w.groupby(['pid','label_ts'])['value'].apply(list).reset_index()
    bvp_feats = []
    for _, r in bvp_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['value'], 'bvp'))
        d.update(spectral_features(r['value'], fs=64, prefix='bvp_spec'))
        a = np.array(r['value'])
        d['bvp_zcr'] = float(np.sum(np.diff(np.sign(np.diff(a)))!=0)/len(a)) if len(a)>1 else 0
        bvp_feats.append(d)
    bvp_feat_df = pd.DataFrame(bvp_feats)

    # EDA + delta
    print(' EDA...')
    eda_agg   = eda_w.groupby(['pid','label_ts'])['value'].apply(list).reset_index()
    eda_b_mean = eda_b.groupby(['pid','label_ts'])['value'].mean().reset_index()
    eda_b_mean.columns = ['pid','label_ts','eda_baseline']
    eda_feats = []
    for _, r in eda_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['value'], 'eda'))
        a = np.array(r['value'], dtype=float)
        if len(a) > 2:
            d['eda_slope'] = float(np.polyfit(np.arange(len(a)), a, 1)[0])
            d['eda_peaks'] = float(np.sum((np.diff(np.sign(np.diff(a))))<-1.9))
        else:
            d['eda_slope'] = d['eda_peaks'] = np.nan
        eda_feats.append(d)
    eda_feat_df = pd.DataFrame(eda_feats)
    eda_feat_df = eda_feat_df.merge(eda_b_mean, on=['pid','label_ts'], how='left')
    eda_feat_df['eda_delta'] = eda_feat_df['eda_mean'] - eda_feat_df['eda_baseline']

    # TEMP + delta
    print(' TEMP...')
    temp_feat_df = temp_w.groupby(['pid','label_ts'])['value'].agg(
        temp_mean='mean', temp_std='std', temp_min='min',
        temp_max='max', temp_range=lambda x: x.max()-x.min()).reset_index()
    temp_b_mean = temp_b.groupby(['pid','label_ts'])['value'].mean().reset_index()
    temp_b_mean.columns = ['pid','label_ts','temp_baseline']
    temp_feat_df = temp_feat_df.merge(temp_b_mean, on=['pid','label_ts'], how='left')
    temp_feat_df['temp_delta'] = temp_feat_df['temp_mean'] - temp_feat_df['temp_baseline']

    # HR + delta
    print(' HR...')
    hr_feat_df = hr_w.groupby(['pid','label_ts'])['value'].agg(
        hr_mean='mean', hr_std='std', hr_min='min',
        hr_max='max', hr_range=lambda x: x.max()-x.min()).reset_index()
    hr_b_mean = hr_b.groupby(['pid','label_ts'])['value'].mean().reset_index()
    hr_b_mean.columns = ['pid','label_ts','hr_baseline']
    hr_feat_df = hr_feat_df.merge(hr_b_mean, on=['pid','label_ts'], how='left')
    hr_feat_df['hr_delta'] = hr_feat_df['hr_mean'] - hr_feat_df['hr_baseline']

    # IBI / HRV
    print(' IBI/HRV...')
    ibi_agg = ibi_w.groupby(['pid','label_ts'])['value'].apply(list).reset_index()
    ibi_feats = []
    for _, r in ibi_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['value'], 'ibi'))
        a = np.array(r['value'], dtype=float); a = a[~np.isnan(a)]
        if len(a) > 1:
            diffs = np.diff(a)
            d['ibi_rmssd'] = float(np.sqrt(np.mean(diffs**2)))
            d['ibi_sdnn']  = float(np.std(a))
            d['ibi_pnn50'] = float(np.mean(np.abs(diffs)>50))
        else:
            d['ibi_rmssd'] = d['ibi_sdnn'] = d['ibi_pnn50'] = np.nan
        d.update(spectral_features(r['value'], fs=4, prefix='ibi_spec'))
        ibi_feats.append(d)
    ibi_feat_df = pd.DataFrame(ibi_feats)

    # Brain
    print(' Brain...')
    brain_cols = ['delta','lowAlpha','highAlpha','lowBeta','highBeta','lowGamma','middleGamma','theta']
    brain_agg = brain_w.groupby(['pid','label_ts'])[brain_cols].agg(['mean','std'])
    brain_agg.columns = [f'brain_{c}_{s}' for c,s in brain_agg.columns]
    brain_agg = brain_agg.reset_index()
    brain_agg['brain_alpha_beta']   = ((brain_agg['brain_lowAlpha_mean']+brain_agg['brain_highAlpha_mean']) /
                                        (brain_agg['brain_lowBeta_mean']+brain_agg['brain_highBeta_mean']+1e-9))
    brain_agg['brain_theta_alpha']  = (brain_agg['brain_theta_mean'] /
                                        (brain_agg['brain_lowAlpha_mean']+brain_agg['brain_highAlpha_mean']+1e-9))
    brain_agg['brain_engagement']   = (brain_agg['brain_lowBeta_mean'] /
                                        (brain_agg['brain_theta_mean']+brain_agg['brain_lowAlpha_mean']+1e-9))

    # ACC + delta
    print(' ACC...')
    acc_w['mag'] = np.sqrt(acc_w['x']**2 + acc_w['y']**2 + acc_w['z']**2)
    acc_b['mag'] = np.sqrt(acc_b['x']**2 + acc_b['y']**2 + acc_b['z']**2)
    acc_agg   = acc_w.groupby(['pid','label_ts'])['mag'].apply(list).reset_index()
    acc_b_mean = acc_b.groupby(['pid','label_ts'])['mag'].mean().reset_index()
    acc_b_mean.columns = ['pid','label_ts','acc_baseline']
    acc_feats = []
    for _, r in acc_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['mag'], 'acc'))
        a = np.array(r['mag'], dtype=float)
        if len(a) > 2:
            jerk = np.diff(a)
            d['acc_jerk_mean'] = float(np.mean(np.abs(jerk)))
            d['acc_jerk_std']  = float(np.std(jerk))
        else:
            d['acc_jerk_mean'] = d['acc_jerk_std'] = np.nan
        acc_feats.append(d)
    acc_feat_df = pd.DataFrame(acc_feats)
    acc_feat_df = acc_feat_df.merge(acc_b_mean, on=['pid','label_ts'], how='left')
    acc_feat_df['acc_delta'] = acc_feat_df['acc_mean'] - acc_feat_df['acc_baseline']

    # Merge all
    print(' Merging...')
    base = labels_df[['id','pid','timestamp']].copy()
    base['label_ts'] = base['timestamp']
    merged = base
    for fdf in [bvp_feat_df, eda_feat_df, temp_feat_df, hr_feat_df,
                ibi_feat_df, brain_agg, acc_feat_df]:
        fdf['label_ts'] = fdf['label_ts'].astype(np.int64)
        merged = merged.merge(fdf, on=['pid','label_ts'], how='left')
    return merged.drop(columns=['label_ts'])

print('build_features() defined.')

build_features() defined.


In [9]:
print('Extracting TRAIN features...')
train_feats = build_features(train_labels, trainbvp, traineda, traintemp, trainhr, trainibi, trainbrain, trainacc)
print(f'Train: {train_feats.shape}')

Extracting TRAIN features...
 Forward windows...
 Lookback baseline...
 BVP...
 EDA...
 TEMP...
 HR...
 IBI/HRV...
 Brain...
 ACC...
 Merging...
Train: (1456, 102)


In [10]:
print('Extracting TEST features...')
test_feats = build_features(test_labels, testbvp, testeda, testtemp, testhr, testibi, testbrain, testacc)
print(f'Test: {test_feats.shape}')

Extracting TEST features...
 Forward windows...
 Lookback baseline...
 BVP...
 EDA...
 TEMP...
 HR...
 IBI/HRV...
 Brain...
 ACC...
 Merging...
Test: (1496, 102)


## 4. Lag Features (same as v4: lag-1, lag-2)

In [11]:
LAG_COLS = [
    'eda_mean','eda_min','eda_q75','eda_slope','eda_delta',
    'temp_mean','temp_min','temp_delta',
    'hr_mean','hr_std','hr_delta',
    'bvp_mean','bvp_skew',
    'acc_mean','acc_max','acc_delta',
    'ibi_rmssd','ibi_sdnn',
]

def add_lag_features(feat_df, lag_cols, lags=[1,2]):
    parts = []
    for pid in feat_df['pid'].unique():
        sub = feat_df[feat_df['pid']==pid].sort_values('timestamp').copy()
        for lag in lags:
            for col in lag_cols:
                if col in sub.columns:
                    sub[f'{col}_lag{lag}'] = sub[col].shift(lag)
        parts.append(sub)
    return pd.concat(parts).sort_values('id').reset_index(drop=True)

print('Adding lag features...')
train_w_lags = add_lag_features(train_feats, LAG_COLS)
test_w_lags  = add_lag_features(test_feats,  LAG_COLS)
print(f'Train: {train_w_lags.shape}')

Adding lag features...
Train: (1456, 138)


## 5. Missing Indicators + Imputation

In [12]:
train_df = train_w_lags.merge(train_labels[['id','arousal']], on='id', how='left')
test_df  = test_w_lags.copy()

feat_cols = [c for c in train_df.columns if c not in ['id','pid','timestamp','arousal']]

sensor_groups = {
    'bvp':   [c for c in feat_cols if c.startswith('bvp')],
    'eda':   [c for c in feat_cols if c.startswith('eda')],
    'temp':  [c for c in feat_cols if c.startswith('temp')],
    'hr':    [c for c in feat_cols if c.startswith('hr')],
    'ibi':   [c for c in feat_cols if c.startswith('ibi')],
    'brain': [c for c in feat_cols if c.startswith('brain')],
    'acc':   [c for c in feat_cols if c.startswith('acc')],
}
for grp, cols in sensor_groups.items():
    if cols:
        ind = f'{grp}_missing'
        train_df[ind] = train_df[cols].isnull().any(axis=1).astype(int)
        test_df[ind]  = test_df[cols].isnull().any(axis=1).astype(int)

feat_cols = [c for c in train_df.columns if c not in ['id','pid','timestamp','arousal']]
indicator_cols = [f'{g}_missing' for g in sensor_groups]

def impute_by_pid(df, cols):
    df = df.copy()
    df[cols] = df[cols].fillna(df.groupby('pid')[cols].transform('median'))
    df[cols] = df[cols].fillna(df[cols].median())
    return df

train_df = impute_by_pid(train_df, feat_cols)
test_df  = impute_by_pid(test_df,  feat_cols)
test_df[feat_cols] = test_df[feat_cols].fillna(train_df[feat_cols].median())

print(f'Feature count: {len(feat_cols)}')
print(f'Train NaN: {train_df[feat_cols].isnull().sum().sum()}')

Feature count: 142
Train NaN: 8736


## 6. V7 Interaction Features

Person-independent cross-sensor arousal signals:
- EDA × HR delta: both increase with arousal, product amplifies signal
- HRV HF power: parasympathetic withdrawal = arousal marker
- Brain alpha suppression: alpha decreases with arousal

In [13]:
def add_interaction_features(df):
    df = df.copy()
    # EDA x HR: both increase with sympathetic activation
    df['eda_hr_interaction'] = df['eda_delta'].fillna(0) * df['hr_delta'].fillna(0)
    # Arousal index: normalized EDA + HR combined signal
    df['arousal_index'] = (df['eda_delta'].fillna(0) + df['hr_delta'].fillna(0)) / 2.0
    # IBI HF power (high = calm, low = aroused)
    if 'ibi_spec_hf' in df.columns:
        df['hrv_calm'] = df['ibi_spec_hf'].fillna(0)  # rename for clarity
    # Temp drop: skin temp decreases with arousal (vasoconstriction)
    df['temp_drop'] = -df['temp_delta'].fillna(0)  # flip sign: positive = temp dropped = aroused
    # Brain alpha suppression: low alpha = high arousal
    if 'brain_lowAlpha_mean' in df.columns:
        df['alpha_suppression'] = -df['brain_lowAlpha_mean'].fillna(0)
    return df

train_df = add_interaction_features(train_df)
test_df  = add_interaction_features(test_df)

# Update feature columns
feat_cols = [c for c in train_df.columns if c not in ['id','pid','timestamp','arousal']]
print(f'Features after interactions: {len(feat_cols)}')

Features after interactions: 147


## 7. Outlier Detection

In [14]:
X_all = train_df[feat_cols].values
y_all = train_df['arousal'].values
outlier_mask = np.zeros(len(train_df), dtype=bool)
for cls in np.unique(y_all):
    idx = np.where(y_all == cls)[0]
    contamination = 0.05 if len(idx) < 100 else 0.07
    iso = IsolationForest(n_estimators=100, contamination=contamination, random_state=42)
    preds = iso.fit_predict(X_all[idx])
    n_out = np.sum(preds == -1)
    print(f' Class {cls} ({len(idx)}): {n_out} outliers')
    outlier_mask[idx[preds==-1]] = True
train_clean = train_df[~outlier_mask].copy().reset_index(drop=True)
print(f'Kept: {len(train_clean)} / {len(train_df)}')

 Class 1 (55): 3 outliers
 Class 2 (430): 31 outliers
 Class 3 (554): 39 outliers
 Class 4 (345): 25 outliers
 Class 5 (72): 4 outliers
Kept: 1354 / 1456


## 8. Per-Person Normalisation

In [15]:
norm_cols = [c for c in feat_cols if not c.endswith('_missing')]

def zscore_by_pid(df, cols):
    df = df.copy()
    for pid in df['pid'].unique():
        mask = df['pid'] == pid
        sub  = df.loc[mask, cols]
        mu   = sub.mean()
        sig  = sub.std().replace(0, np.nan)
        df.loc[mask, cols] = (sub - mu) / sig
    df[cols] = df[cols].fillna(0)
    return df

train_norm = zscore_by_pid(train_clean, norm_cols)
test_norm  = zscore_by_pid(test_df,     norm_cols)
print('Normalisation done.')
print(f'Train NaN: {train_norm[feat_cols].isnull().sum().sum()}')

Normalisation done.
Train NaN: 0


## 9. Feature Selection — top 60

In [16]:
X_sel = train_norm[feat_cols].values
y_sel = train_norm['arousal'].values.astype(float)

rf_sel = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_sel.fit(X_sel, y_sel)

imp_df = pd.DataFrame({'feature': feat_cols, 'importance': rf_sel.feature_importances_}).sort_values('importance', ascending=False)

N_KEEP = 60
top_features = imp_df.head(N_KEEP)['feature'].tolist()
for ind in indicator_cols:
    if ind not in top_features and ind in feat_cols:
        top_features.append(ind)

print(f'Selected {len(top_features)} features')
print('\nTop 15:')
print(imp_df.head(15)[['feature','importance']].to_string(index=False))

Selected 67 features

Top 15:
      feature  importance
    acc_count    0.075508
    eda_count    0.071489
    ibi_count    0.067631
    bvp_count    0.053540
      eda_min    0.036232
  hr_baseline    0.034132
 eda_min_lag1    0.029891
eda_mean_lag2    0.023164
temp_baseline    0.022942
     ibi_skew    0.018294
hr_delta_lag2    0.015577
 eda_min_lag2    0.012864
     temp_min    0.011526
 acc_baseline    0.010402
 eda_q75_lag1    0.009925


## 10. ADASYN Oversampling + Class Weights

In [17]:
X_train = train_norm[top_features].values
y_train = train_norm['arousal'].values

unique, counts = np.unique(y_train, return_counts=True)
print('Before resampling:')
for u, c in zip(unique, counts): print(f'  Class {u}: {c}')

target = {cls: max(cnt, 150) for cls, cnt in zip(unique.astype(int), counts)}
try:
    res = ADASYN(sampling_strategy=target, n_neighbors=min(4, min(counts)-1), random_state=42)
    X_res, y_res = res.fit_resample(X_train, y_train)
    print('ADASYN succeeded.')
except Exception as e:
    print(f'ADASYN failed, using SMOTE...')
    res = SMOTE(sampling_strategy=target, k_neighbors=min(3, min(counts)-1), random_state=42)
    X_res, y_res = res.fit_resample(X_train, y_train)

class_counts_res = dict(zip(*np.unique(y_res, return_counts=True)))
total_res = len(y_res)
class_weights = {cls: total_res/(len(class_counts_res)*cnt) for cls,cnt in class_counts_res.items()}
class_weights[1] = class_weights[1] * 2.0
class_weights[5] = class_weights[5] * 2.0
sample_weights = np.array([class_weights[y] for y in y_res])

print('\nClass weights:')
for k,v in sorted(class_weights.items()): print(f'  Class {k}: {v:.3f}')

Before resampling:
  Class 1: 52
  Class 2: 399
  Class 3: 515
  Class 4: 320
  Class 5: 68
ADASYN succeeded.

Class weights:
  Class 1: 3.836
  Class 2: 0.774
  Class 3: 0.600
  Class 4: 0.965
  Class 5: 4.145


## 11. LOPO Cross-Validation (RF + LGB + XGB)

In [18]:
def ordinal_clip(pred):
    return np.clip(np.round(pred), 1, 5).astype(int)

rf_params = dict(
    n_estimators=600, max_depth=10, min_samples_leaf=4,
    min_samples_split=8, max_features='sqrt', max_samples=0.8,
    random_state=42, n_jobs=-1
)
lgb_params = dict(
    objective='regression_l1', metric='mae',
    n_estimators=400, learning_rate=0.04, num_leaves=31, max_depth=5,
    min_child_samples=20, subsample=0.75, colsample_bytree=0.75,
    reg_alpha=0.5, reg_lambda=1.0, random_state=42, verbose=-1
)
xgb_params = dict(
    objective='reg:absoluteerror',
    n_estimators=400, learning_rate=0.04, max_depth=5,
    min_child_weight=20, subsample=0.75, colsample_bytree=0.75,
    reg_alpha=0.5, reg_lambda=1.0, random_state=42, verbosity=0, n_jobs=-1
)

pids = train_norm['pid'].unique()
lopo_maes = {'rf': [], 'lgb': [], 'xgb': [], 'ens': []}
lopo_pid_results = []
print(f'LOPO CV ({len(pids)} folds)...\n')

for held_pid in pids:
    mask_val = train_norm['pid'] == held_pid
    mask_tr  = ~mask_val
    X_tr  = train_norm.loc[mask_tr,  top_features].values
    y_tr  = train_norm.loc[mask_tr,  'arousal'].values.astype(float)
    X_val = train_norm.loc[mask_val, top_features].values
    y_val = train_norm.loc[mask_val, 'arousal'].values

    u_tr, c_tr = np.unique(y_tr, return_counts=True)
    fold_target = {cls: max(cnt, 100) for cls,cnt in zip(u_tr.astype(int), c_tr)}
    safe_k = max(1, min(3, min(c_tr)-1))
    try:
        sm = ADASYN(sampling_strategy=fold_target, n_neighbors=safe_k, random_state=42)
        X_tr_s, y_tr_s = sm.fit_resample(X_tr, y_tr)
    except:
        sm = SMOTE(sampling_strategy=fold_target, k_neighbors=safe_k, random_state=42)
        X_tr_s, y_tr_s = sm.fit_resample(X_tr, y_tr)

    sw = np.array([class_weights.get(int(y), 1.0) for y in y_tr_s])

    m_rf  = RandomForestRegressor(**rf_params)
    m_rf.fit(X_tr_s, y_tr_s, sample_weight=sw)

    m_lgb = lgb.LGBMRegressor(**lgb_params)
    m_lgb.fit(X_tr_s, y_tr_s, sample_weight=sw)

    m_xgb = xgb.XGBRegressor(**xgb_params)
    m_xgb.fit(X_tr_s, y_tr_s, sample_weight=sw)

    p_rf  = m_rf.predict(X_val)
    p_lgb = m_lgb.predict(X_val)
    p_xgb = m_xgb.predict(X_val)

    # Find best 3-way weights
    best_mae, best_w = float('inf'), (0.5, 0.3, 0.2)
    for w1 in [0.4, 0.5, 0.6, 0.7]:
        for w2 in [0.1, 0.2, 0.3, 0.4]:
            w3 = 1 - w1 - w2
            if w3 < 0: continue
            p = ordinal_clip(w1*p_rf + w2*p_lgb + w3*p_xgb)
            m = mean_absolute_error(y_val, p)
            if m < best_mae:
                best_mae, best_w = m, (w1, w2, w3)

    mae_rf  = mean_absolute_error(y_val, ordinal_clip(p_rf))
    mae_lgb = mean_absolute_error(y_val, ordinal_clip(p_lgb))
    mae_xgb = mean_absolute_error(y_val, ordinal_clip(p_xgb))
    mae_ens = best_mae

    lopo_maes['rf'].append(mae_rf)
    lopo_maes['lgb'].append(mae_lgb)
    lopo_maes['xgb'].append(mae_xgb)
    lopo_maes['ens'].append(mae_ens)
    lopo_pid_results.append({'pid': held_pid, 'n': mask_val.sum(), 'rf': mae_rf,
                              'lgb': mae_lgb, 'xgb': mae_xgb, 'ens': mae_ens, 'best_w': best_w})
    print(f'  [{held_pid}] RF={mae_rf:.3f} LGB={mae_lgb:.3f} XGB={mae_xgb:.3f} ENS={mae_ens:.3f} w={best_w}')

print()
for name, maes in lopo_maes.items():
    print(f'{name.upper():6s} LOPO MAE: {np.mean(maes):.4f} ± {np.std(maes):.4f}')
print(f'\nv4 ENS LOPO: 0.8002 | v7 ENS LOPO: {np.mean(lopo_maes["ens"]):.4f}')

LOPO CV (11 folds)...

  [01Z2] RF=0.891 LGB=1.101 XGB=1.067 ENS=0.908 w=(0.7, 0.1, 0.20000000000000004)
  [70N8] RF=0.449 LGB=0.566 XGB=0.566 ENS=0.478 w=(0.7, 0.3, 5.551115123125783e-17)
  [7PF3] RF=0.830 LGB=0.912 XGB=1.000 ENS=0.728 w=(0.6, 0.4, 0.0)
  [CQ2G] RF=1.074 LGB=1.355 XGB=1.380 ENS=1.091 w=(0.7, 0.2, 0.10000000000000003)
  [D1XP] RF=0.155 LGB=0.411 XGB=0.372 ENS=0.194 w=(0.7, 0.1, 0.20000000000000004)
  [DT5C] RF=1.016 LGB=1.139 XGB=1.230 ENS=0.844 w=(0.4, 0.1, 0.5)
  [F1ZM] RF=0.800 LGB=0.962 XGB=0.990 ENS=0.800 w=(0.7, 0.1, 0.20000000000000004)
  [LIUY] RF=1.460 LGB=1.508 XGB=1.532 ENS=1.460 w=(0.6, 0.1, 0.30000000000000004)
  [SE4Q] RF=0.607 LGB=0.820 XGB=0.820 ENS=0.656 w=(0.7, 0.1, 0.20000000000000004)
  [TPQI] RF=0.737 LGB=0.695 XGB=0.800 ENS=0.705 w=(0.4, 0.4, 0.19999999999999996)
  [Y21H] RF=0.871 LGB=0.947 XGB=0.985 ENS=0.909 w=(0.7, 0.1, 0.20000000000000004)

RF     LOPO MAE: 0.8082 ± 0.3237
LGB    LOPO MAE: 0.9469 ± 0.3098
XGB    LOPO MAE: 0.9765 ± 0.3210
ENS  

## 12. Final Training (all data)

In [19]:
print('Training final models...')

# Compute LOPO-inverse weights for final ensemble
inv_rf  = 1.0 / np.mean(lopo_maes['rf'])
inv_lgb = 1.0 / np.mean(lopo_maes['lgb'])
inv_xgb = 1.0 / np.mean(lopo_maes['xgb'])
total_inv = inv_rf + inv_lgb + inv_xgb
w_rf  = inv_rf  / total_inv
w_lgb = inv_lgb / total_inv
w_xgb = inv_xgb / total_inv
print(f'Final weights: RF={w_rf:.3f}, LGB={w_lgb:.3f}, XGB={w_xgb:.3f}')

rf_final  = RandomForestRegressor(**rf_params)
rf_final.fit(X_res, y_res.astype(float), sample_weight=sample_weights)
print(' RF done.')

lgb_final = lgb.LGBMRegressor(**lgb_params)
lgb_final.fit(X_res, y_res.astype(float), sample_weight=sample_weights)
print(' LGB done.')

xgb_final = xgb.XGBRegressor(**xgb_params)
xgb_final.fit(X_res, y_res.astype(float), sample_weight=sample_weights)
print(' XGB done.')

# Train sanity
p_train = ordinal_clip(w_rf*rf_final.predict(X_train) + w_lgb*lgb_final.predict(X_train) + w_xgb*xgb_final.predict(X_train))
print(f'Train MAE: {mean_absolute_error(y_train, p_train):.4f}')

Training final models...
Final weights: RF=0.373, LGB=0.318, XGB=0.309
 RF done.
 LGB done.
 XGB done.
Train MAE: 0.3028


## 13. Predict + Calibrate

**Keep v4 rank-based calibration** — removing it caused v6 to drop from 0.23→0.20

In [20]:
X_test = test_norm[top_features].values
raw_rf  = rf_final.predict(X_test)
raw_lgb = lgb_final.predict(X_test)
raw_xgb = xgb_final.predict(X_test)
raw_ens = w_rf * raw_rf + w_lgb * raw_lgb + w_xgb * raw_xgb

# Rank-based calibration (KEEP — critical for accuracy)
train_y_float = train_norm['arousal'].values.astype(float)
train_q = np.percentile(train_y_float, np.linspace(0, 100, 1000))
pred_q  = np.percentile(raw_ens,       np.linspace(0, 100, 1000))
calib_fn = interp1d(pred_q, train_q, bounds_error=False, fill_value=(train_q[0], train_q[-1]))
pred_classes = ordinal_clip(calib_fn(raw_ens))

print('Prediction distribution:')
u, c = np.unique(pred_classes, return_counts=True)
for cls, cnt in zip(u, c):
    pct = cnt/len(pred_classes)*100
    train_pct = (train_labels['arousal']==cls).mean()*100
    print(f'  Class {cls}: {cnt:4d} ({pct:.1f}%) | Train {train_pct:.1f}% | Δ={pct-train_pct:+.1f}%')

Prediction distribution:
  Class 1:   57 (3.8%) | Train 3.8% | Δ=+0.0%
  Class 2:  441 (29.5%) | Train 29.5% | Δ=-0.1%
  Class 3:  569 (38.0%) | Train 38.0% | Δ=-0.0%
  Class 4:  354 (23.7%) | Train 23.7% | Δ=-0.0%
  Class 5:   75 (5.0%) | Train 4.9% | Δ=+0.1%


## 14. Save Submission

In [21]:
submission = pd.DataFrame({'id': test_labels['id'].values, 'arousal': pred_classes})
assert len(submission) == 1496
assert submission['arousal'].between(1,5).all()
assert (submission['id'] == test_labels['id']).all()

submission.to_csv('submission_v7.csv', index=False)

print('Final distribution:')
print(submission['arousal'].value_counts().sort_index())
print('\nSaved: submission_v7.csv')
print(f'\nv4 LB: 0.23191 | v6 LB: 0.20604')
print(f'v7 LOPO ENS: {np.mean(lopo_maes["ens"]):.4f} (v4 was 0.8002)')
print(f'If LOPO improved → expect LB improvement')

Final distribution:
arousal
1     57
2    441
3    569
4    354
5     75
Name: count, dtype: int64

Saved: submission_v7.csv

v4 LB: 0.23191 | v6 LB: 0.20604
v7 LOPO ENS: 0.7975 (v4 was 0.8002)
If LOPO improved → expect LB improvement
